# Giá trị hoá đơn: phân phối đuôi dài, mean vs median, trục log

Bộ `online-retail-ii`: giao dịch của một cửa hàng quà tặng online ở Anh, bán cả cho khách lẻ lẫn khách
sỉ. Bài này đọc sheet **`Year 2009-2010`** (525 461 dòng, 01/12/2009 tới 09/12/2010); lý thuyết module
dùng sheet năm sau, nên con số ở đây là của riêng bạn. Mỗi dòng là một mặt hàng trong một hoá đơn:
`Invoice`, `StockCode`, `Quantity`, `Price` (GBP/đơn vị), `InvoiceDate`, `Customer ID`, `Country`.

Head of Sales hỏi: **"Một hoá đơn điển hình của mình đáng bao nhiêu tiền?"** và muốn một chart cho slide.
Câu hỏi về **phân phối** của một biến số, nên chart mặc định là histogram. Nhưng giá trị hoá đơn lệch
phải rất mạnh, và histogram trục tuyến tính sẽ cho bạn thấy gần như không gì cả.

Việc của bạn: tính giá trị mỗi hoá đơn đúng quy tắc làm sạch, tóm tắt phân phối bằng 5 con số, tạo
biên bin trên thang log, vẽ 4 cách nhìn cùng một phân phối, rồi **làm phần trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/08-visualization-chart-reading/exercises/basic/03-order-value-distribution -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2009-2010 (xlsx: khoảng 20-60 giây)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

RETAIL = ensure("online-retail-ii")
tx = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2009-2010")
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())
tx.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Giá trị mỗi hoá đơn

`invoice_totals(tx) -> pd.Series`: tổng `Quantity * Price` của mỗi hoá đơn, index là mã hoá đơn dạng
**chuỗi** (tên index `Invoice`). Quy tắc làm sạch, áp dụng trên từng **dòng** trước khi cộng:

- bỏ hoá đơn huỷ: mã bắt đầu bằng `C`;
- bỏ dòng `Quantity <= 0` (trả hàng, điều chỉnh kho) và dòng `Price <= 0` (hàng tặng, điều chỉnh nợ xấu
  mã `A...` có giá âm).

Excel đọc mã thuần số thành `int` và mã có chữ thành `str` trong cùng một cột, nên đổi cả cột sang `str`
trước khi xét tiền tố `C`. Hoá đơn còn ít nhất một dòng hợp lệ thì có mặt trong kết quả.

In [ ]:
def invoice_totals(tx: pd.DataFrame) -> pd.Series:
    """Tổng Quantity * Price mỗi hoá đơn hợp lệ, index là mã hoá đơn dạng str (tên "Invoice")."""
    # TODO: invoice = tx["Invoice"].astype(str)
    # TODO: giữ dòng không bắt đầu bằng "C", Quantity > 0, Price > 0
    # TODO: cộng Quantity * Price theo invoice
    raise NotImplementedError

In [ ]:
# explore: bao nhiêu hoá đơn còn lại sau khi làm sạch?
inv = invoice_totals(tx)
code = tx["Invoice"].astype(str)
print(f"{code.nunique():,} mã hoá đơn, trong đó {code[code.str.startswith('C')].nunique():,} mã huỷ")
print(f"{len(inv):,} hoá đơn hợp lệ, tổng {inv.sum():,.0f} GBP")
inv.describe()

## 2. Tóm tắt phân phối bằng 5 con số

`summary_stats(s) -> dict` với đúng các khoá `mean`, `median`, `p90`, `p99`, `skew` (giá trị `float`).
Dùng mặc định của pandas: `quantile` nội suy tuyến tính, `skew` là hệ số lệch mẫu đã hiệu chỉnh.

Phân phối đối xứng có `mean ≈ median` và `skew ≈ 0`. `skew` lớn và `mean` cao hơn hẳn `median` là dấu
hiệu đuôi phải dài: vài giá trị rất lớn kéo trung bình lên.

In [ ]:
def summary_stats(s: pd.Series) -> dict[str, float]:
    """mean, median, p90, p99, skew của s (mặc định của pandas), giá trị là float."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: 5 con số, và hai câu hỏi một DA nên tự hỏi khi thấy mean >> median
stats = summary_stats(inv)
print({k: round(v, 2) for k, v in stats.items()})
print(f"mean cao hơn median {stats['mean'] / stats['median'] - 1:.0%}")
print(f"tỉ lệ hoá đơn lớn hơn mean: {(inv > stats['mean']).mean():.1%}")
top1 = inv[inv >= stats["p99"]]
print(f"1% hoá đơn lớn nhất ({len(top1)} hoá đơn) đóng góp {top1.sum() / inv.sum():.1%} doanh thu")

## 3. Biên bin cách đều trên thang log

Histogram trục tuyến tính với giá trị từ 0.19 tới gần 50 000 GBP dồn gần như mọi hoá đơn vào một hai
cột đầu tiên. Cách sửa: bin cách đều trên **log10** (mỗi bin rộng gấp một hệ số cố định so với bin
trước), rồi vẽ trục x thang log.

`log_bins(s, n) -> np.ndarray`: `n + 1` biên bin, cách đều trên log10 từ `min(s)` tới `max(s)`, sao
cho `np.histogram(s, bins=edges)` đếm đủ mọi giá trị. Thang log không xác định với số `<= 0`: gặp giá trị
như vậy thì `raise ValueError`.

In [ ]:
def log_bins(s: pd.Series, n: int) -> np.ndarray:
    """n + 1 biên bin cách đều trên log10, từ min(s) tới max(s); ValueError nếu có giá trị <= 0."""
    # TODO: kiểm tra giá trị dương
    # TODO: np.logspace(np.log10(min), np.log10(max), n + 1); biên đầu/cuối đúng bằng min/max
    raise NotImplementedError

In [ ]:
# explore: 4 cách nhìn cùng một phân phối
fig, ax = plt.subplots(2, 2, figsize=(12, 7))

ax[0, 0].hist(inv, bins=50, color="#9aa5b1")
ax[0, 0].set_title("1. Histogram trục tuyến tính, 50 bin đều", loc="left")
ax[0, 0].set_xlabel("GBP/hoá đơn")

ax[0, 1].hist(inv, bins=log_bins(inv, 50), color="#4c78a8")
ax[0, 1].set_xscale("log")
for name, ls in [("median", "-"), ("mean", "--"), ("p99", ":")]:
    ax[0, 1].axvline(stats[name], color="#f58518", ls=ls, lw=1.2)
    ax[0, 1].annotate(f"{name} {stats[name]:,.0f}", (stats[name], 0.97), xycoords=("data", "axes fraction"),
                      rotation=90, va="top", ha="right", fontsize=8)
ax[0, 1].set_title("2. Histogram bin log, trục x log", loc="left")
ax[0, 1].set_xlabel("GBP/hoá đơn (thang log)")

# boxplot tính tứ phân vị và râu trên log10 của giá trị: râu 1.5 IQR tính trên thang tuyến tính
# rồi vẽ lên trục log sẽ lệch hẳn một bên
ax[1, 0].boxplot(np.log10(inv), orientation="horizontal", widths=0.5, flierprops={"markersize": 2, "alpha": 0.3})
ax[1, 0].set_yticks([])
ax[1, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{10 ** v:g}"))
ax[1, 0].set_title("3. Boxplot trên log10", loc="left")
ax[1, 0].set_xlabel("GBP/hoá đơn (thang log)")

x = np.sort(inv.to_numpy())
ax[1, 1].plot(x, np.arange(1, len(x) + 1) / len(x), color="#4c78a8")
ax[1, 1].set_xscale("log")
for q in (0.5, 0.9, 0.99):
    ax[1, 1].axhline(q, color="#c9ced6", lw=0.8)
ax[1, 1].set_title("4. ECDF: tỉ lệ hoá đơn có giá trị ≤ x", loc="left")
ax[1, 1].set_xlabel("GBP/hoá đơn (thang log)")

for a in ax.flat:
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Hoá đơn điển hình khoảng {stats['median']:,.0f} GBP; mean {stats['mean']:,.0f} bị vài đơn sỉ kéo lên",
             x=0.01, ha="left")
fig.tight_layout()
plt.show()
print(f"ECDF tại 1 000 GBP: {(inv <= 1000).mean():.1%} hoá đơn")

## 4. Đọc chart, rồi làm trắc nghiệm

Đọc 4 chart theo quy trình 5 bước. Gợi ý cho từng bước:

1. **Trục và đơn vị**: chart nào trục tuyến tính, chart nào log? Trên trục log, khoảng cách từ 10 tới
   100 bằng từ 100 tới 1 000.
2. **Hình dạng**: trên thang log, phân phối có gần đối xứng không? Có một đỉnh hay nhiều đỉnh?
3. **Điểm bất thường**: hoá đơn lớn nhất là ai, có phải lỗi nhập liệu? Hoá đơn nhỏ nhất 0.19 GBP là gì?
   ECDF có đoạn nào dốc đứng bất thường không? Phóng to quanh chỗ đó bằng cell dưới.
4. **So với tham chiếu**: median nằm ở đâu so với mean và p99?
5. **Câu hỏi tiếp theo**: tách khách sỉ và khách lẻ (theo `Customer ID`, `Country`, số lượng mỗi dòng)?

Head of Sales cần **một** chart cho slide: chọn chart nào trong 4, headline là gì?

Xong thì mở app, làm phần **trắc nghiệm** của bài.

In [ ]:
# explore: phóng to 200-400 GBP, bin 10 GBP
zoom = pd.cut(inv, np.arange(200, 410, 10)).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar([f"{iv.left:.0f}" for iv in zoom.index], zoom.values, color="#9aa5b1")
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("cận dưới của bin (GBP/hoá đơn)")
ax.set_title("Số hoá đơn theo bin 10 GBP, khoảng 200-400 GBP", loc="left")
plt.show()
zoom.to_string()